# Experimental Analysis

Using the script `sampling_distribution.py -p=f -e=E` we find the first semimersenne prime with $2^f | (p + 1)$ and set $e = \lceil \log(p)/e \rceil + E + 2$.   

Then we iterate over all possible odd $0 \leq y \leq 2^{e-1} / \sqrt{p}$ and sample a corresponding ideal, if it exists. 

We then reduce the binary quadratic form associated to the ideal and store it with the value of $y$ used. 
If two or more $y$'s are found to define the same form we store them together and increment the number of collisions.

In [2]:
import json
from utilities.class_number_formula import approx_class_number
from ideal2d import sample_ideal2d
from math import comb
import pandas as pd
from math import comb

In [3]:
# loading data, will take a while
data_stored = {}
folder_path = Path("data/heu-test")

for json_file in folder_path.glob("*.json"):
    with open(json_file, "r") as f:
        data = json.load(f)
    data_stored[json_file] = data

In [4]:
data_by_prime = dict()
p = 1

# organising data by base prime
primes = []
for fn, data in data_stored.items():
    p = data["prime"]
    p2 = (p+1).valuation(2)
    if not p in primes:
        data_by_prime[p2] = {
                            # computing the class number take a while for p ~ 2^64
                             'class_num' : approx_class_number(fundamental_discriminant(-p)),
                             'prime'     : data["prime"] 
                            }
        primes.append(p)
    data_by_prime[p2][data['exponent']] = data

In [5]:
# our data contain the following bit sizes
sorted(list(data_by_prime.keys()))

[32, 43, 55, 64]

Here we give some functions that allow to compute expected behaviour of our data.

In [6]:
def exp_collisions(n: int, t: int):
    """
    Returns the expected number of collisions when drawing t times with
    replacement in n class.

    Formula:
    E[#collisions] = t - E[#distinct classes seen]
                   = t - C_1
                   = t - n * ( 1 - ( 1 -  1 / n)^t)
    """
    exp_colli = t - numerical_approx(n * ( 1 - ( 1 -  1 / n)^t))
    return round(exp_colli,1)

def expected_counts_exact_k(n: int, t: int, k: int, rounding = True):
    """
    Returns E[# of classes seen exactly k times]
    when drawing t times with replacement in n class (uniformly).
    
    Formula:
        C_3(k) = n * C(t, k) * (1/n)^k * (1 - 1/n)^(t - k)
    """
    if not (0 <= k <= t):
        return None
    if n <= 0 or t < 0:
        raise ValueError("n must be >= 1 and t must be >= 0.")

    p = 1. / n
    one_minus_p = 1. - p
    out = n * comb(t, k) * (p ** k) * (one_minus_p ** (t - k))
    if rounding:
        return round(out, 1)
    return out

def prob_of_zero_coll(n: int, t: int, rounding = True):
    """
    Returns the probability of seeing zero collisions when drawing t times with
    replacement in n class (uniformly).

    Formula:
    P[#collisions = 0] = C_2

    """
    out = exp(- t*(t-1) / (2. * n))
    if not rounding:
        return out
    return round(out, 2)

def expected_support(p,e):
    """
    Returns the upper bound for the size of the set I_e defined in the paper,
    """
    out = 0.76 * 2**(e-2) / sqrt(p * 2 * (e-1) * log(2.) ) 
    return round(out)

### Printing the data

Using the following function we can compute the expected datas and combine them with the actual one.

The columns contain the following data:
- $e$: the exponent used in the experiment
- exp. success: the expected size of $I_e$
- succ: the actual size of $I_e$ in the experiment
- supp: the actual size of the support of the distribution (i.e. the number of distinct classes found)
- actual colls: the actual number of collisions found in the experiment
- exp colls $C_1$: the expected number of collisions according to $C_1$
- prob 0 colls: the probability of seeing zero collisions according to $C_2$
- exp. $C_3(k)$: the expected number of classes seen exactly $k$ times according to $C_3(k)$
- actual $C_3(k)$: the actual number of classes seen exactly $k$ times in the experiment

these last two columns are repeated for $k = 1, 2, 3$.

Observe that the expected datas are in line with the actual ones, and that the
probability of seeing zero collisions is coherent with the actual number of
collisions found in the experiment.
Note that the expected size of $I_e$ slightly underestimates the actual size
found in the experiment. 

In [7]:
def getdf(dataset):
    df = pd.DataFrame(columns=["$e$", "exp. success", "succ", "supp", "actual colls", "exp colls $C_1$",  "prob 0 colls", 
                               "exp. $C_3(1)$", "actual $C_3(1)$", 
                               "exp. $C_3(2)$", "actual $C_3(2)$", 
                               "exp. $C_3(3)$", "actual $C_3(3)$", 
                              ])

    for key, data in dataset.items():
        if key == 'class_num':
            class_number = data
            continue
        if key == 'prime':
            p = data
            continue
        e = key
        size_y = 2 ** (e - 2) // isqrt(p) - 1

        exp_supp = expected_support(p,e)
        succ = data['successes']
        exp_colli = exp_collisions(class_number, succ)
        actual_colli = data['collisions']
        p_val_zero_coll = prob_of_zero_coll(class_number, succ)
        row = [e, exp_supp, succ, len(data['classes']), actual_colli, exp_colli,  p_val_zero_coll]

        classes = data['classes']
        histogram = {}
        for v in classes.values():
            histogram[len(v)] = histogram.get(len(v), 0) + 1
        for k in range(1,4):
            row.append(expected_counts_exact_k(class_number, succ, k))
            row.append(0 if histogram.get(k) is None else histogram.get(k))

        df.loc[len(df)] = row
    df.sort_values(by=['$e$'], inplace=True)
    return df



In [8]:
print("Data for prime log p ~ 64")
prime2 = 64
df = getdf(data_by_prime[prime2])
df

Data for prime log p ~ 64


,$e$,exp. success,succ,supp,actual colls,exp colls $C_1$,prob 0 colls,exp. $C_3(1)$,actual $C_3(1)$,exp. $C_3(2)$,actual $C_3(2)$,exp. $C_3(3)$,actual $C_3(3)$
6,40,4,8,8,0,0.0,1.0,8.0,8,0.0,0,0.0,0
3,45,115,328,328,0,0.0,1.0,328.0,328,0.0,0,0.0,0
4,50,3489,4596,4596,0,0.0,1.0,4596.0,4596,0.0,0,0.0,0
2,51,6908,9142,9142,0,0.0,0.99,9142.0,9142,0.0,0,0.0,0
1,52,13680,17397,17397,0,0.0,0.98,17397.0,17397,0.0,0,0.0,0
5,53,27095,34328,34328,0,0.1,0.91,34327.8,34328,0.1,0,0.0,0
8,54,53677,38549,38548,1,0.1,0.89,38548.8,38547,0.1,1,0.0,0
7,55,106355,266824,266816,8,5.5,0.0,266813.1,266808,5.5,8,0.0,0
0,56,210768,260109,260106,3,5.2,0.01,260098.6,260103,5.2,3,0.0,0
9,57,417754,510332,510319,13,20.0,0.0,510292.0,510306,20.0,13,0.0,0


In [9]:
print("Data for prime log p ~ 43")
prime2 = 43
df = getdf(data_by_prime[prime2])
df

Data for prime log p ~ 43


,$e$,exp. success,succ,supp,actual colls,exp colls $C_1$,prob 0 colls,exp. $C_3(1)$,actual $C_3(1)$,exp. $C_3(2)$,actual $C_3(2)$,exp. $C_3(3)$,actual $C_3(3)$
11,35,185,297,297,0,0.0,0.99,297.0,297,0.0,0,0.0,0
4,36,365,510,510,0,0.0,0.97,509.9,510,0.0,0,0.0,0
3,37,720,1029,1029,0,0.1,0.87,1028.7,1029,0.1,0,0.0,0
13,38,1420,2117,2117,0,0.6,0.55,2115.8,2117,0.6,0,0.0,0
2,39,2802,4015,4012,3,2.1,0.12,4010.7,4009,2.1,3,0.0,0
5,40,5531,7783,7778,5,8.0,0.0,7767.0,7773,8.0,5,0.0,0
8,41,10922,15259,15229,30,30.7,0.0,15197.6,15199,30.7,30,0.0,0
6,42,21577,29698,29582,116,116.3,0.0,29465.8,29466,115.7,116,0.3,0
14,43,42637,58084,57673,411,443.7,0.0,57198.9,57264,439.1,407,2.2,2
9,44,84276,112571,110941,1630,1658.5,0.0,109270.4,109322,1625.9,1608,16.1,11


In [10]:
print("Data for prime log p ~ 55")
prime2 = 55
df = getdf(data_by_prime[prime2])
df

Data for prime log p ~ 55


,$e$,exp. success,succ,supp,actual colls,exp colls $C_1$,prob 0 colls,exp. $C_3(1)$,actual $C_3(1)$,exp. $C_3(2)$,actual $C_3(2)$,exp. $C_3(3)$,actual $C_3(3)$
1,40,86,264,264,0,0.0,1.0,264.0,264,0.0,0,0.0,0
0,45,2604,8164,8164,0,0.1,0.89,8163.8,8164,0.1,0,0.0,0
2,47,10185,30166,30166,0,1.5,0.22,30163.0,30166,1.5,0,0.0,0
4,48,20153,58524,58522,2,5.7,0.0,58512.6,58520,5.7,2,0.0,0
3,50,78948,67301,67295,6,7.5,0.0,67285.9,67289,7.5,6,0.0,0


In [11]:
print("Data for prime log p ~ 32")
prime2 = 32
df = getdf(data_by_prime[prime2])
df

Data for prime log p ~ 32


,$e$,exp. success,succ,supp,actual colls,exp colls $C_1$,prob 0 colls,exp. $C_3(1)$,actual $C_3(1)$,exp. $C_3(2)$,actual $C_3(2)$,exp. $C_3(3)$,actual $C_3(3)$
10,28,57,69,69,0,0.0,0.98,69.0,69,0.0,0,0.0,0
1,29,112,117,117,0,0.1,0.93,116.9,117,0.1,0,0.0,0
5,30,220,436,434,2,1.0,0.37,434.0,432,1.0,2,0.0,0
2,31,432,441,439,2,1.0,0.36,439.0,437,1.0,2,0.0,0
6,32,849,874,870,4,4.0,0.02,866.1,866,3.9,4,0.0,0
9,33,1672,1638,1630,8,13.9,0.0,1610.4,1622,13.7,8,0.1,0
4,34,3293,3153,3097,56,51.1,0.0,3051.3,3041,50.0,56,0.5,0
11,35,6489,6159,5961,198,193.0,0.0,5777.0,5765,185.0,194,3.9,2
3,36,12791,11745,11084,661,688.8,0.0,10394.9,10451,634.7,605,25.8,28
8,37,25224,11119,10512,607,618.6,0.0,9905.1,9926,572.5,565,22.1,21


## Chi Square test for uniform distribution
In this experiment we just run the sampling algorithm for various values of
$p$ and $e$ and we store the number of times each class is hit and by which
$y$'s.
Obtained with the script `sampling_distribution.py -p=f -e=E -r`

The data frame contains the following columns:
- $p$: the prime used in the experiment
- $e$: the exponent used in the experiment
- $\#$ keys: the number of distinct classes found in the experiment
- $\#$ colls: the number of collisions found in the experiment
- $\chi^2$: the value of the chi square statistic for the observed distribution
- p-value: the p-value of the chi square test for the observed distribution

Observe that as long as the number of collision is small, the distribution is
not far from uniform, as expected. If not we see a significant deviation from
the uniform distribution.

In [12]:
from scipy.stats import chisquare
import re
from scipy.stats import chisquare
import re
from pathlib import Path
import json
import pandas as pd

In [13]:
df = pd.DataFrame(columns = ['$p$', '$e$', '$\#$ keys', '$\#$ colls', '$\chi^2$', 'p-value'])
folder_path = Path("data/unif-test")

for json_file in folder_path.glob("*.json"):
    with open(json_file, 'r') as f:
        data = json.load(f)
    p = data['prime']
    e = data["exponent"]
    aggregated = {key: sum(inner_val.values()) for key, inner_val in data['classes'].items()}
    
    chi2_stat, p_value = chisquare(list(aggregated.values()))
    pr = re.sub(r'(\d+)\^(\d+) \* (\d+)', r'$\1^{\2} \\cdot \3 - 1$', str(factor(p+1)))
    row = [pr, e, len(aggregated), data['collisions'], round(chi2_stat,1), round(p_value,3)]
    df.loc[len(df)] = row

df.sort_values(by=['$p$', '$e$'], inplace=True)
df['p-value'] = df['p-value'].map('{:.3f}'.format)
df
# print(df.to_latex(multicolumn=True, escape=False, index=False))

,$p$,$e$,$\#$ keys,$\#$ colls,$\chi^2$,p-value
6,$2^{32} \cdot 5 - 1$,28,69,0,67.9,0.479
17,$2^{32} \cdot 5 - 1$,33,1629,8,1659.4,0.288
18,$2^{32} \cdot 5 - 1$,34,3097,56,3572.2,0.000
2,$2^{32} \cdot 5 - 1$,35,5960,198,7632.7,0.000
9,$2^{32} \cdot 5 - 1$,36,11084,661,17358.5,0.000
14,$2^{32} \cdot 5 - 1$,37,20503,2618,45827.4,0.000
1,$2^{32} \cdot 5 - 1$,38,35760,8776,119718.3,0.000
8,$2^{32} \cdot 5 - 1$,39,57274,29705,313186.4,0.000
0,$2^{32} \cdot 5 - 1$,40,79451,89078,696900.7,0.000
7,$2^{43} \cdot 3 - 1$,38,2117,0,2023.6,0.924
